# CLINICAL-CORE: Ingestión y Procesamiento Multimodal
Este notebook orquestación la ejecución del pipeline desde Google Colab, integrando el código del repositorio con tus datos en Google Drive.

## 1. Preparar Entorno de Ejecución

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# @title Configuración de Rutas base
REPO_PATH = "/content/clinical_core" # @param {type:"string"}
DRIVE_DATA_ROOT = "/content/drive/MyDrive/data_tesis" # @param {type:"string"}

In [ ]:
# Clonar el repositorio si no existe, o actualizarlo
import os
if not os.path.exists(REPO_PATH):
    !git clone https://github.com/raulprtech/clinical_core.git {REPO_PATH}
else:
    %cd {REPO_PATH}
    !git pull

## 2. Instalar Dependencias (Core Stack)

In [ ]:
# Instalamos las librerías necesarias definidas en el repositorio
!pip install -q -r {REPO_PATH}/requirements.txt
print("✅ Dependencias instaladas correctamente.")

## 3. Configurar Experimento Local

In [ ]:
import sys
import yaml
from pathlib import Path

# Añadir la carpeta 'code' al path para permitir imports
sys.path.append(str(Path(REPO_PATH) / "code"))

CODE_DIR = Path(REPO_PATH) / "code"
XML_DATA_DIR = Path(DRIVE_DATA_ROOT) / "clinicalsupplement"
RESULTS_DIR = Path(DRIVE_DATA_ROOT) / "results"

# Cargar y actualizar configuración del experimento
EXP_CONFIG_PATH = CODE_DIR / "experiments/experiment_config_tabular_only.yaml"

with open(EXP_CONFIG_PATH) as f:
    cfg = yaml.safe_load(f)

# Sincronizar rutas con Google Drive
cfg['data']['xml_dir'] = str(XML_DATA_DIR)
cfg['output']['base_dir'] = str(RESULTS_DIR)

with open(EXP_CONFIG_PATH, 'w') as f:
    yaml.dump(cfg, f, sort_keys=False, default_flow_style=False)

print(f"🚀 Configuración lista para experimento: {cfg['experiment']['name']}")
print(f"📍 Datos: {cfg['data']['xml_dir']}")
print(f"💾 Resultados: {cfg['output']['base_dir']}")

## 4. Ejecutar el Pipeline

In [ ]:
from utils.experiment_runner import run_experiment

# Asegurarnos de que estamos en el directorio de código para que los paths relativos funcionen
os.chdir(str(CODE_DIR))

# Ejecución total
summary = run_experiment("experiments/experiment_config_tabular_only.yaml")

## 5. Visualizar Resumen de Resultados

In [ ]:
import json
from IPython.display import JSON

print("Resumen de la ejecución:")
print(f"Cindex promedio (Cox): {summary.get('phases', {}).get('phase_2', {}).get('mean', {}).get('cox_baseline', 'N/A')}")
print(f"Cindex promedio (TabPFN): {summary.get('phases', {}).get('phase_2', {}).get('mean', {}).get('tabpfn', 'N/A')}")

JSON(summary)